<a href="https://colab.research.google.com/github/Minh-Khuong/Bai-tap-nhom-Ai-tuan-2/blob/main/Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-fuzzy ipyleaflet geopy

from google.colab import output
output.enable_custom_widget_manager()

import requests
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from ipyleaflet import Map, Marker, Polyline, Icon, TileLayer
import ipywidgets as widgets
from IPython.display import display
import random
from geopy.geocoders import Nominatim

WEATHER_API_KEY = '996ac29028ae56015ed48f8a6317ae14'
TOMTOM_API_KEY = 'JrHK7EaBpDs9t9APeTuPTlmT3iQ5At3J'

DRIVERS_DB = [
    {"name": "Nguyễn Minh A", "vehicle": "Honda Vario 150", "plate": "59-S2 999.99", "rating": "5.0 ⭐"},
    {"name": "Võ Trí T", "vehicle": "VinFast Evo 200", "plate": "59-E1 555.55", "rating": "4.9 ⭐"},
    {"name": "Nguyễn Thành N", "vehicle": "Honda Wave Alpha", "plate": "59-P1 345.67", "rating": "4.8 ⭐"},
    {"name": "Lê Huỳnh Đ", "vehicle": "Yamaha Sirius", "plate": "59-X1 112.23", "rating": "4.7 ⭐"},
    {"name": "Phạm Hoàng S", "vehicle": "Honda Air Blade", "plate": "59-K2 888.88", "rating": "4.9 ⭐"}
]

geolocator = Nominatim(user_agent="fuzzy_ride_peak_final")

def get_weather_value(city='Ho Chi Minh'):
    weather_values_map = {
        'clear sky': 10, 'few clouds': 10, 'scattered clouds': 10, 'broken clouds': 10, 'overcast clouds': 9.5,
        'light rain': 6, 'moderate rain': 5, 'heavy intensity rain': 4, 'very heavy rain': 3, 'extreme rain': 2, 'freezing rain': 3,
        'light intensity shower rain': 3.5, 'shower rain': 3, 'heavy intensity shower rain': 2,
        'light intensity drizzle': 3.5, 'drizzle': 3, 'heavy intensity drizzle': 2, 'heavy intensity drizzle rain': 2, 'drizzle rain': 2,
        'thunderstorm with light rain': 1.5, 'thunderstorm with rain': 1, 'thunderstorm with heavy rain': 1, 'thunderstorm': 1.5,
        'heavy thunderstorm': 1.5, 'ragged thunderstorm': 1
    }
    try:
        url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={WEATHER_API_KEY}&units=metric"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            weather_desc = response.json()['weather'][0]['description']
            return weather_values_map.get(weather_desc, 5), weather_desc
    except:
        pass
    return 10, "Trời đẹp"

def calculate_grab_fare(distance, traffic_val, weather_val):
    base_price = 12500 if distance <= 2 else 12500 + (distance - 2) * 4300

    Traffic = ctrl.Antecedent(np.arange(0, 101, 1), 'Traffic')
    Weather = ctrl.Antecedent(np.arange(1, 10.5, 0.5), 'Weather')
    Surge = ctrl.Consequent(np.arange(0.9, 2.6, 0.1), 'Surge')

    Traffic['L'] = fuzz.trimf(Traffic.universe, [0, 0, 40])
    Traffic['M'] = fuzz.trimf(Traffic.universe, [30, 50, 70])
    Traffic['H'] = fuzz.trimf(Traffic.universe, [60, 100, 100])

    Weather['B'] = fuzz.trimf(Weather.universe, [1, 1, 4])
    Weather['M'] = fuzz.trimf(Weather.universe, [3, 6, 8])
    Weather['G'] = fuzz.trimf(Weather.universe, [7, 10, 10])

    Surge['Normal'] = fuzz.trimf(Surge.universe, [0.9, 1.0, 1.1])
    Surge['High'] = fuzz.trimf(Surge.universe, [1.1, 1.3, 1.6])
    Surge['VeryHigh'] = fuzz.trimf(Surge.universe, [1.5, 2.0, 2.5])

    rules = [
        ctrl.Rule(Traffic['L'] & Weather['G'], Surge['Normal']),
        ctrl.Rule(Traffic['L'] & Weather['M'], Surge['Normal']),
        ctrl.Rule(Traffic['L'] & Weather['B'], Surge['High']),
        ctrl.Rule(Traffic['M'] & Weather['G'], Surge['Normal']),
        ctrl.Rule(Traffic['M'] & Weather['M'], Surge['High']),
        ctrl.Rule(Traffic['M'] & Weather['B'], Surge['VeryHigh']),
        ctrl.Rule(Traffic['H'] & Weather['G'], Surge['High']),
        ctrl.Rule(Traffic['H'] & Weather['M'], Surge['VeryHigh']),
        ctrl.Rule(Traffic['H'] & Weather['B'], Surge['VeryHigh'])
    ]

    surge_ctrl = ctrl.ControlSystem(rules)
    surge_system = ctrl.ControlSystemSimulation(surge_ctrl)
    surge_system.input['Traffic'] = traffic_val
    surge_system.input['Weather'] = weather_val
    try:
        surge_system.compute()
        multiplier = surge_system.output['Surge']
    except:
        multiplier = 1.0

    return base_price * multiplier

NEON_GREEN = '#00FF66'
GRAB_BLUE = '#4285F4'
WARN_YELLOW = '#F2C94C'
DARK_BG = '#0B0D10'

start_coords, end_coords = None, None
start_marker, end_marker, route_line = None, None, None
current_distance, current_traffic = 0, 30

def render_css():
    return f"""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@500;700;800;900&family=Inter:wght@400;500;600;700&display=swap');
    @keyframes subtleGlow {{ 0% {{ box-shadow: 0 0 5px rgba(0,255,102,0.2); }} 50% {{ box-shadow: 0 0 15px rgba(0,255,102,0.5); }} 100% {{ box-shadow: 0 0 5px rgba(0,255,102,0.2); }} }}
    .premium-font {{ font-family: 'Montserrat', sans-serif; }}
    .inter {{ font-family: 'Inter', sans-serif; }}
    .panel-base {{ background: rgba(255, 255, 255, 0.03); backdrop-filter: blur(12px); border-radius: 16px; padding: 18px; margin-bottom: 12px; border: 1px solid rgba(255,255,255,0.05); transition: all 0.3s ease; }}
    .dist-panel {{ border-left: 4px solid {NEON_GREEN}; box-shadow: -4px 0 20px rgba(0,255,102,0.1); }}
    .status-panel-blue {{ border: 1px solid rgba(66,133,244,0.3); background: rgba(66,133,244,0.05); }}
    .status-panel-green {{ border: 1px solid rgba(0,255,102,0.3); background: rgba(0,255,102,0.05); }}
    .status-panel-warn {{ border: 1px solid rgba(242,201,76,0.3); background: rgba(242,201,76,0.05); }}
    .price-text {{ font-size: 52px; font-weight: 900; color: {NEON_GREEN}; text-shadow: 0 0 15px rgba(0,255,102,0.4); margin: 5px 0 15px 0; letter-spacing: -1px; }}
    .price-zero {{ font-size: 52px; font-weight: 900; color: #333; margin: 5px 0 15px 0; letter-spacing: -1px; }}
    .driver-card {{ background: rgba(0,0,0,0.4); border-radius: 16px; padding: 20px; margin-top: 15px; border: 1px solid rgba(0,255,102,0.3); box-shadow: 0 8px 32px rgba(0,0,0,0.3), inset 0 0 20px rgba(0,255,102,0.05); animation: subtleGlow 2.5s infinite; }}
    .tag-neon {{ background-color: rgba(0, 255, 102, 0.1); color: {NEON_GREEN}; padding: 6px 14px; border-radius: 8px; font-weight: 800; font-size: 14px; border: 1px solid rgba(0,255,102,0.5); font-family: 'Montserrat', sans-serif; letter-spacing: 1px; }}
    </style>
    """

def update_dist_ui(km="0.00"):
    color = NEON_GREEN if km != "0.00" else "#555"
    return f"""<div class='panel-base dist-panel'>
        <div class='inter' style='color:#888; font-size:12px; font-weight:600; text-transform:uppercase; letter-spacing:1px; margin-bottom:4px;'>🛣️ Khoảng cách lộ trình</div>
        <div class='premium-font' style='color:{color}; font-size:26px; font-weight:800;'>{km} <span style='font-size:14px; color:#666; font-weight:600;'>KM</span></div>
    </div>"""

def update_status_ui(msg, state="blue"):
    color_map = {"blue": GRAB_BLUE, "green": NEON_GREEN, "warn": WARN_YELLOW}
    color = color_map.get(state, GRAB_BLUE)
    icon = "✨" if state=="green" else "⏳" if state=="warn" else "📍"
    return f"""<div class='panel-base status-panel-{state}' style='padding: 14px 16px;'>
        <div class='inter' style='color:{color}; font-size:14px; font-weight:500;'>{icon} {msg}</div>
    </div>"""

def update_price_ui(price_str="0"):
    css_class = "price-text" if price_str != "0" else "price-zero"
    unit_color = "#888" if price_str != "0" else "#222"
    return f"""<div>
        <div class='inter' style='color:#666; font-size:12px; font-weight:700; letter-spacing:1.5px; text-transform:uppercase;'>Cước phí AI đề xuất</div>
        <h1 class='premium-font {css_class}'>{price_str} <span style='font-size:20px; color:{unit_color}; font-weight:600; text-shadow:none;'>VNĐ</span></h1>
    </div>"""

css_injection = widgets.HTML(value=render_css())

txt_start = widgets.Text(placeholder='📍 Điểm đón (VD: UEH Nguyễn Tri Phương)', layout=widgets.Layout(width='100%', margin='0 0 10px 0'))
txt_end = widgets.Text(placeholder='🏁 Điểm đến (VD: Chợ Bến Thành)', layout=widgets.Layout(width='100%', margin='0 0 10px 0'))
btn_search = widgets.Button(description='ĐẶT THEO ĐỊA CHỈ NÀY', layout=widgets.Layout(width='100%', height='45px', margin='0 0 15px 0'))
btn_search.style.button_color = '#1A1D21'
btn_search.style.text_color = '#aaa'

input_box = widgets.VBox([
    widgets.HTML(value="<div class='inter' style='color:#777; font-size:11px; font-weight:700; letter-spacing:1px; margin-bottom:10px;'>BẠN MUỐN ĐI ĐÂU?</div>"),
    txt_start, txt_end, btn_search
], layout=widgets.Layout(width='100%'))

dist_lbl = widgets.HTML(value=update_dist_ui())
status_lbl = widgets.HTML(value=update_status_ui("Xin chào! Hãy gõ địa chỉ hoặc click lên bản đồ để bắt đầu chuyến đi.", "blue"))
price_lbl = widgets.HTML(value=update_price_ui())

btn_calc = widgets.Button(description='TÍNH CƯỚC NGAY', layout=widgets.Layout(width='100%', height='55px', margin='10px 0 5px 0'))
btn_calc.style.button_color = '#1A1D21'
btn_calc.style.text_color = '#555'
btn_calc.style.font_weight = 'bold'

btn_book = widgets.Button(description='🚀 GỌI TÀI XẾ BIKER', layout=widgets.Layout(width='100%', height='55px', margin='5px 0', display='none'))
btn_book.style.button_color = NEON_GREEN
btn_book.style.text_color = '#000000'
btn_book.style.font_weight = '900'

driver_info_area = widgets.HTML(value="")

sidebar = widgets.VBox([
    css_injection,
    widgets.HTML(value=f"<h1 class='premium-font' style='color:{NEON_GREEN}; margin: 0 0 25px 0; font-size:34px; letter-spacing:1px; text-shadow: 0 0 15px rgba(0,255,102,0.3);'>FUZZY RIDE</h1>"),
    input_box,
    dist_lbl,
    status_lbl,
    widgets.HTML(value="<hr style='border-color: rgba(255,255,255,0.05); margin: 25px 0;'>"),
    price_lbl,
    btn_calc,
    btn_book,
    driver_info_area
], layout=widgets.Layout(width='36%', padding='35px', border='none', background_color=DARK_BG))

google_map_layer = TileLayer(url="https://mt0.google.com/vt/lyrs=m&hl=vi&x={x}&y={y}&z={z}&s=Ga", max_zoom=20)
m = Map(layers=(google_map_layer,), center=(10.7631, 106.6678), zoom=15, layout=widgets.Layout(width='64%', height='800px'))

icon_start = Icon(icon_url='https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-2x-green.png', shadow_url='https://cdnjs.cloudflare.com/ajax/libs/leaflet/0.7.7/images/marker-shadow.png', icon_size=[25, 41], icon_anchor=[12, 41])
icon_end = Icon(icon_url='https://raw.githubusercontent.com/pointhi/leaflet-color-markers/master/img/marker-icon-2x-red.png', shadow_url='https://cdnjs.cloudflare.com/ajax/libs/leaflet/0.7.7/images/marker-shadow.png', icon_size=[25, 41], icon_anchor=[12, 41])

def reset_all_ui():
    global current_distance, route_line, start_coords, end_coords
    current_distance = 0

    dist_lbl.value = update_dist_ui()
    price_lbl.value = update_price_ui()

    driver_info_area.value = ""

    btn_book.layout.display = 'none'
    btn_book.disabled = False
    btn_book.description = '🚀 GỌI TÀI XẾ BIKER'
    btn_book.style.button_color = NEON_GREEN
    btn_book.style.text_color = '#000000'

    btn_calc.style.button_color = '#1A1D21'
    btn_calc.style.text_color = '#555'
    btn_calc.disabled = False

def draw_real_route():
    global route_line, current_distance, current_traffic
    status_lbl.value = update_status_ui("Đang phân tích lộ trình nhanh nhất từ vệ tinh...", "warn")

    driver_info_area.value = ""
    btn_book.layout.display = 'none'

    latA, lonA = start_coords
    latB, lonB = end_coords
    url = f"https://api.tomtom.com/routing/1/calculateRoute/{latA},{lonA}:{latB},{lonB}/json?key={TOMTOM_API_KEY}&traffic=true"

    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        if 'routes' in data and len(data['routes']) > 0:
            route = data['routes'][0]
            points = route['legs'][0]['points']
            path_coords = [(p['latitude'], p['longitude']) for p in points]

            if route_line in m.layers: m.remove_layer(route_line)
            route_line = Polyline(locations=path_coords, color=GRAB_BLUE, fill=False, weight=6, opacity=0.9)
            m.add_layer(route_line)

            min_lat, max_lat = min(p[0] for p in path_coords), max(p[0] for p in path_coords)
            min_lon, max_lon = min(p[1] for p in path_coords), max(p[1] for p in path_coords)
            m.fit_bounds([[min_lat, min_lon], [max_lat, max_lon]])

            current_distance = route['summary']['lengthInMeters'] / 1000
            dist_lbl.value = update_dist_ui(f"{current_distance:.2f}")

            duration_in_traffic = route['summary'].get('travelTimeInSeconds')
            duration_without_traffic = route['summary'].get('noTrafficTravelTimeInSeconds', duration_in_traffic)
            traffic_ratio = duration_in_traffic / duration_without_traffic if duration_without_traffic > 0 else 1

            if traffic_ratio >= 1.5: current_traffic = 90
            elif traffic_ratio >= 1.2: current_traffic = 60
            else: current_traffic = 30

            status_lbl.value = update_status_ui("Lộ trình đã sẵn sàng. Bạn có thể xem cước phí!", "green")
            btn_calc.style.button_color = '#2A2E33'
            btn_calc.style.text_color = NEON_GREEN
    except:
        status_lbl.value = update_status_ui("Rất tiếc, hệ thống không thể tìm thấy đường đi.", "warn")

def search_address_logic(b=None):
    global start_coords, end_coords, start_marker, end_marker

    status_lbl.value = update_status_ui("Đang định vị vị trí của bạn...", "blue")

    try:
        if txt_start.value and txt_start.value != "📍 Đã ghim trên bản đồ":
            locA = geolocator.geocode(txt_start.value + ", Ho Chi Minh, Vietnam")
            if locA:
                start_coords = (locA.latitude, locA.longitude)
                if start_marker in m.layers: m.remove_layer(start_marker)
                start_marker = Marker(location=start_coords, icon=icon_start, draggable=False)
                m.add_layer(start_marker)
                m.center = start_coords

        if txt_end.value and txt_end.value != "🏁 Đã ghim trên bản đồ":
            locB = geolocator.geocode(txt_end.value + ", Ho Chi Minh, Vietnam")
            if locB:
                end_coords = (locB.latitude, locB.longitude)
                if end_marker in m.layers: m.remove_layer(end_marker)
                end_marker = Marker(location=end_coords, icon=icon_end, draggable=False)
                m.add_layer(end_marker)
                m.center = end_coords

        if start_coords and end_coords:
            draw_real_route()
        elif start_coords:
            reset_all_ui()
            status_lbl.value = update_status_ui("Vị trí đón đã chốt. Bạn muốn đến đâu?", "blue")
        elif end_coords:
            reset_all_ui()
            status_lbl.value = update_status_ui("Hệ thống đã nhận điểm đến, hãy nhập điểm đón nhé!", "blue")
        else:
            status_lbl.value = update_status_ui("Bạn quên nhập địa chỉ kìa!", "warn")

    except:
        status_lbl.value = update_status_ui("Mạng lưới định vị đang gián đoạn, thử lại nhé.", "warn")

btn_search.on_click(search_address_logic)
txt_start.on_submit(search_address_logic)
txt_end.on_submit(search_address_logic)

def on_map_click(**kwargs):
    global start_coords, end_coords, start_marker, end_marker, route_line, current_distance
    if kwargs.get('type') == 'click':
        coords = kwargs.get('coordinates')

        if start_coords is not None and end_coords is not None:
            m.remove_layer(start_marker)
            m.remove_layer(end_marker)
            if route_line in m.layers: m.remove_layer(route_line)
            reset_all_ui()
            start_coords = None
            end_coords = None

        if start_coords is None:
            reset_all_ui()
            start_coords = coords
            start_marker = Marker(location=coords, icon=icon_start, draggable=False)
            m.add_layer(start_marker)
            txt_start.value = "📍 Đã ghim trên bản đồ"
            status_lbl.value = update_status_ui("Vị trí đón đã chốt. Bạn muốn đến đâu?", "blue")
        elif end_coords is None:
            end_coords = coords
            end_marker = Marker(location=coords, icon=icon_end, draggable=False)
            m.add_layer(end_marker)
            txt_end.value = "🏁 Đã ghim trên bản đồ"
            draw_real_route()

m.on_interaction(on_map_click)

def calculate_fare(b):
    if current_distance == 0: return

    weather_val, weather_desc = get_weather_value()
    final_price = calculate_grab_fare(current_distance, current_traffic, weather_val)

    price_lbl.value = update_price_ui(format(int(final_price), ','))
    status_lbl.value = update_status_ui(f"Cước phí đã tối ưu theo giao thông và thời tiết ({weather_desc}).", "green")

    btn_calc.style.button_color = '#111'
    btn_calc.style.text_color = '#333'

    btn_book.disabled = False
    btn_book.description = '🚀 GỌI TÀI XẾ BIKER'
    btn_book.style.button_color = NEON_GREEN
    btn_book.style.text_color = '#000000'
    btn_book.layout.display = 'block'

def start_booking(b):
    btn_calc.disabled = True
    btn_book.disabled = True

    driver = random.choice(DRIVERS_DB)
    eta = random.randint(2, 6)

    btn_book.description = "✅ TÀI XẾ ĐÃ NHẬN CUỐC"
    btn_book.style.button_color = '#ffffff'

    driver_html = f"""
    <div class="driver-card">
        <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid rgba(255,255,255,0.1); padding-bottom: 15px; margin-bottom: 15px;">
            <div style="display: flex; align-items: center;">
                <div style="width: 48px; height: 48px; border-radius: 50%; background: #000; border: 2px solid {NEON_GREEN}; display: flex; align-items: center; justify-content: center; font-size: 24px; margin-right: 15px; box-shadow: 0 0 10px rgba(0, 255, 102, 0.3);">
                    👤
                </div>
                <div>
                    <div class="premium-font" style="color: white; font-weight: 800; font-size: 16px; text-transform: uppercase;">{driver['name']}</div>
                    <div class="inter" style="color: {NEON_GREEN}; font-size: 12px; margin-top: 5px; font-weight: 600; letter-spacing: 1px;">{driver['rating']}</div>
                </div>
            </div>
            <div class="tag-neon">{driver['plate']}</div>
        </div>
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span class="inter" style="color: #bbb; font-size: 14px; font-weight: 500;">🏍️ {driver['vehicle']}</span>
            <span class="inter" style="color: #000; font-weight: 800; font-size: 13px; background: {NEON_GREEN}; padding: 6px 12px; border-radius: 4px; box-shadow: 0 0 10px rgba(0,255,102,0.4);">Đón trong {eta} phút</span>
        </div>
    </div>
    """
    status_lbl.value = update_status_ui(f"Bác tài đang hoàn tất chuyến đi trước. Xin vui lòng chờ!", "blue")
    driver_info_area.value = driver_html

btn_calc.on_click(calculate_fare)
btn_book.on_click(start_booking)

app_layout = widgets.HBox([m, sidebar], layout=widgets.Layout(width='100%', align_items='stretch'))
display(app_layout)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 81.6 MB/s eta 0:00:00
